In [54]:
import pandas as pd
import matplotlib.pyplot as plt 
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import seaborn as sns
from pathlib import Path
import joblib


In [55]:
pbp_DF = pd.read_csv('../datasets/props_training.csv')
pbp_DF.head()

,game_id,player_id,season,seconds_remaining,current_points,current_minutes,season_ppg,season_fga,season_mpg,fga_so_far,3pa_so_far,season_3pa,current_rebounds,season_rebounds,current_assists,season_assists,score_differential,final_points,final_rebounds,final_assists
0,22200001,203954,2022-23,2751,0,2.15,33.08,20.12,34.61,0,0,3.03,1,10.15,0,4.15,5,26,15,5
1,22200001,203954,2022-23,2689,0,3.18,33.08,20.12,34.61,0,0,3.03,2,10.15,0,4.15,7,26,15,5
2,22200001,203954,2022-23,2689,1,3.18,33.08,20.12,34.61,0,0,3.03,2,10.15,0,4.15,6,26,15,5
3,22200001,203954,2022-23,2673,1,3.45,33.08,20.12,34.61,0,0,3.03,3,10.15,0,4.15,6,26,15,5
4,22200001,203954,2022-23,2632,1,4.13,33.08,20.12,34.61,0,0,3.03,4,10.15,0,4.15,3,26,15,5


In [56]:
pbp_DF['points_remaining'] = pbp_DF['final_points'] - pbp_DF['current_points']
pbp_DF.drop(columns=['game_id', 'player_id', 'season', 'current_rebounds', 'season_rebounds', 'current_assists', 'season_assists', 'final_rebounds', 'final_assists', 'final_points'], inplace=True)
pbp_DF.drop_duplicates(inplace=True)
pbp_DF.head()

,seconds_remaining,current_points,current_minutes,season_ppg,season_fga,season_mpg,fga_so_far,3pa_so_far,season_3pa,score_differential,points_remaining
0,2751,0,2.15,33.08,20.12,34.61,0,0,3.03,5,26
1,2689,0,3.18,33.08,20.12,34.61,0,0,3.03,7,26
2,2689,1,3.18,33.08,20.12,34.61,0,0,3.03,6,25
3,2673,1,3.45,33.08,20.12,34.61,0,0,3.03,6,25
4,2632,1,4.13,33.08,20.12,34.61,0,0,3.03,3,25


In [57]:
feature_cols = ['seconds_remaining', 'current_points', 'current_minutes', 'season_ppg', 'season_fga', 'season_mpg', 'fga_so_far', '3pa_so_far', 'season_3pa', 'score_differential']
X = pbp_DF[feature_cols]
y = pbp_DF['points_remaining']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [58]:
model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=500,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)
model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=True
)

[0]	validation_0-rmse:8.91063
[1]	validation_0-rmse:8.43149
[2]	validation_0-rmse:8.02130
[3]	validation_0-rmse:7.67857
[4]	validation_0-rmse:7.37560
[5]	validation_0-rmse:7.11987
[6]	validation_0-rmse:6.90448
[7]	validation_0-rmse:6.72239
[8]	validation_0-rmse:6.57062
[9]	validation_0-rmse:6.44486
[10]	validation_0-rmse:6.34072
[11]	validation_0-rmse:6.25399
[12]	validation_0-rmse:6.18077
[13]	validation_0-rmse:6.12127
[14]	validation_0-rmse:6.07097
[15]	validation_0-rmse:6.03083
[16]	validation_0-rmse:5.99784
[17]	validation_0-rmse:5.96917
[18]	validation_0-rmse:5.94644
[19]	validation_0-rmse:5.92549
[20]	validation_0-rmse:5.90972
[21]	validation_0-rmse:5.89432
[22]	validation_0-rmse:5.88223
[23]	validation_0-rmse:5.87136
[24]	validation_0-rmse:5.86130
[25]	validation_0-rmse:5.85318
[26]	validation_0-rmse:5.84669
[27]	validation_0-rmse:5.84100
[28]	validation_0-rmse:5.83627
[29]	validation_0-rmse:5.83301
[30]	validation_0-rmse:5.82882
[31]	validation_0-rmse:5.82486
[32]	validation_0-

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.9
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [ ]:
def eval_model():
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R²: {r2:.3f}")

def custom_predictions(custom_data):
    # index=[0] required when custom_data is a dict of scalars (one row)
    custom_df = pd.DataFrame(custom_data, index=[0])
    custom_df = custom_df[['seconds_remaining', 'current_points', 'current_minutes', 'season_ppg', 'season_fga', 'season_mpg', 'fga_so_far', '3pa_so_far', 'season_3pa', 'score_differential']]
    points_remaining = round(model.predict(custom_df)[0])
    points_started = custom_data['current_points']
    print(f"Points started with: {points_started}")
    print(f"Points predicted left to score: {points_remaining}")
    print(f"Final points: {points_started + points_remaining}")

def save_model(model, model_path_name):
    model_path = Path(model_path_name)
    joblib.dump(model, model_path)
    print(f"Model saved to {model_path.absolute()}")
    

In [60]:
eval_model()

MAE: 4.19
RMSE: 32.57
R²: 0.642


In [65]:
custom_data = {
    'seconds_remaining': 720,
    'current_points': 18,
    'current_minutes': 24,
    'season_ppg': 27,
    'season_fga': 19,
    'season_mpg': 36,
    'fga_so_far': 14,
    '3pa_so_far': 8,
    'season_3pa': 10,
    'score_differential': 2,
}

custom_predictions(custom_data)

Points started with: 18
Points predicted left to score: 7
Final points: 25


In [ ]:
save_model(model, 'pts_model.joblib')